In [1]:
import os 
from  langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain.agents import create_agent
from langchain_community.tools import DuckDuckGoSearchRun
from dotenv import load_dotenv
import yfinance as yf
import requests

In [2]:
load_dotenv()

True

In [3]:
model = ChatGroq(model='llama-3.3-70b-versatile')

In [4]:
search_tool = DuckDuckGoSearchRun()

In [5]:
# tool 1 get info about stock and give info about company 

@tool

def get_stock_overview(stock_symbol : str) -> str :
    """  Retrieves a high-level overview of a given stock.

    This tool takes a stock symbol as input and returns essential
    information about the company and its stock, such as current price,
    basic financial details, and general company information. It provides
    a quick snapshot useful for initial analysis.

    Parameters:
        symbol (str): Stock ticker symbol (e.g., "TCS.NS", "AAPL").

    Returns:
        dict: A structured response containing:
              - Company name and sector
              - Current stock price and market cap
              - Basic financial metrics
              - General company information

    Use Case:
        Helps a stock analysis agent quickly understand a company’s profile
        before performing deeper fundamental, technical, or news-based analysis."""
    stock_name = yf.Ticker(stock_symbol)
    result = stock_name.info
    return {
        "name": result.get("longName"),
        "sector": result.get("sector"),
        "industry": result.get("industry"),
        "summary": result.get("longBusinessSummary"),
        "current_price": result.get("currentPrice"),
        "previous_close": result.get("previousClose"),
        "day_high": result.get("dayHigh"),
        "day_low": result.get("dayLow"),
        "market_cap": result.get("marketCap"),
        "52w_high": result.get("fiftyTwoWeekHigh"),
        "52w_low": result.get("fiftyTwoWeekLow"),
        "volume": result.get("volume"),
        "avg_volume": result.get("averageVolume"),
        "recommendation": result.get("recommendationKey"),
        "target_price": result.get("targetMeanPrice")
    }

In [6]:
@tool

def analyze_stock(stock_symbol : str ) -> str:
    """ Performs fundamental and technical analysis of a given stock.

    This tool takes a stock symbol as input and provides key insights
    based on both fundamental data (financial health, valuation metrics)
    and technical indicators (price trends, momentum, patterns). It helps
    in evaluating whether a stock is potentially a good buy, hold, or sell.

    Parameters:
        symbol (str): Stock ticker symbol (e.g., "TCS.NS", "AAPL").

    Returns:
        dict: A structured response containing:
              - Fundamental metrics (PE ratio, market cap, earnings, etc.)
              - Technical indicators (moving averages, RSI, trends, etc.)
              - Overall analysis summary for decision-making

    Use Case:
        Helps a stock analysis agent generate informed investment advice
        by combining financial strength and price movement analysis."""
    stock_name = yf.Ticker(stock_symbol)
    result = stock_name.info 
    hist = stock_name.history(period="3mo")

    close_price = hist["Close"]

    trend = "uptrend" if close_price.iloc[-1] > close_price.mean() else "downtrend"

    return {
        "pe_ratio": result.get("trailingPE"),
        "eps": result.get("trailingEps"),
        "revenue_growth": result.get("revenueGrowth"),
        "profit_margin": result.get("profitMargins"),
        "roe": result.get("returnOnEquity"),
        "debt_to_equity": result.get("debtToEquity"),
        "free_cashflow": result.get("freeCashflow"),
        "recent_prices": close_price.tail(10).tolist(),
        "trend": trend
    }

In [ ]:
@tool

def get_news(q, searchIn, from_date, to_date, sortBy, language):
    """ This tool is designed to support stock analysis by retrieving recent and relevant
    news, which can help in making informed investment decisions. The date range can
    be adjusted dynamically to analyze short-term or long-term news trends.

    Parameters:
        q (str): Search query (e.g., company or stock name like "Infosys", "TCS").
        searchIn (str): Fields to search in (e.g., "title", "description", "content").
        from_date (str): Start date for news (format: "YYYY-MM-DD").
        to_date (str): End date for news (format: "YYYY-MM-DD").
        sortBy (str): Sorting method ("relevancy", "popularity", "publishedAt").
        language (str): Language of news articles (e.g., "en").

    Returns:
        dict: JSON response containing news articles, including title, source,
              publication date, and URL.

    Use Case:
        Helps a stock analysis agent understand market sentiment, recent events,
        and news trends affecting a company before giving investment advice.
    """
    url = "https://newsapi.org/v2/everything"

    params = {
        "q": q,
        "searchIn": searchIn,
        "from": from_date,
        "to": to_date,
        "sortBy": sortBy,
        "language": language,
        "apiKey": os.environ["NEWS_API"]
    }

    response = requests.get(url, params=params)
    
    return response.json()

In [8]:
agent = create_agent(
    model = model ,
    tools= [get_stock_overview ,analyze_stock , get_news , search_tool],
    system_prompt="""
You are a helpful financial assistant that provides stock prices,
basic investment insights, and analysis using multiple tools.

Capabilities:
You can use the following tools to give better advice:
- Stock Overview Tool → Provides basic company info and key metrics
- Analysis Tool → Provides fundamental and technical analysis
- News Tool → Provides recent news affecting the stock

Rules:
- Always use Yahoo Finance ticker symbols (e.g., TCS.NS, AAPL).
- Do NOT use search tools for stock prices.
- Map company names correctly to ticker symbols before processing.
- Use available tools when needed to enhance your response.

Your Tasks:
1. Identify the stock/company from the user query.
2. Convert it to the correct Yahoo Finance ticker symbol.
3. Fetch stock data (price, trend, etc.).
4. Optionally use:
   - Overview → for basic company understanding
   - Analysis → for deeper insights
   - News → for recent developments
5. Provide a clear and helpful response.

Response Format (VERY IMPORTANT):

 **Stock Summary**
- **Ticker:** <ticker>
- **Current Price:** ₹<price>
- **Trend:** Increasing / Decreasing

 **Quick Insight**
- 1–2 line simple explanation of what’s happening

 **Recent News (if used)**
- Short headline or summary (optional)

 **Analysis (if used)**
- Key fundamental or technical insight (optional)

 **Advice**
- **Buy / Hold / Sell**
- Reason in simple words (not financial advice)

Guidelines:
- Keep the response clean, readable, and well-spaced.
- Use bullet points and emojis for clarity.
- Avoid technical jargon unless necessary.
- Keep explanations short and practical.
- Always include the ticker symbol.
"""
)

In [11]:
response = agent.invoke({"messages": [{"role": "user", "content": "give me technicalk analysis of suzlon" }]})
response

{'messages': [HumanMessage(content='give me technicalk analysis of suzlon', additional_kwargs={}, response_metadata={}, id='1cc11156-a28c-416e-a890-14f0e3edfa70'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'nqw22q41c', 'function': {'arguments': '{"stock_symbol":"SUZLON.NS"}', 'name': 'analyze_stock'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 1931, 'total_tokens': 1952, 'completion_time': 0.070788183, 'completion_tokens_details': None, 'prompt_time': 0.295866658, 'prompt_tokens_details': None, 'queue_time': 0.092212596, 'total_time': 0.366654841}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d9c46-9308-7d13-9b1e-1a87261f642c-0', tool_calls=[{'name': 'analyze_stock', 'args': {'stock_symbol': 'SUZLON.NS'}, 'id': 'nqw22q41c', 'type': 'tool_call'}], invalid_

In [12]:
print(response['messages'][-1].content)

**Stock Summary**
- **Ticker:** SUZLON.NS
- **Current Price:** ₹6.45
- **Trend:** Increasing

**Quick Insight**
- Suzlon's stock is currently in an uptrend, with a recent price increase.
- The company's revenue growth and profit margin are positive, but the debt-to-equity ratio is high.

**Recent News (if used)**
- No recent news available.

**Analysis (if used)**
- Suzlon's PE ratio is 22.43, indicating that the stock may be overvalued.
- The company's EPS is 2.36, which is a positive sign.
- The debt-to-equity ratio is 5.05, which is high and may indicate a higher risk for investors.

**Advice**
- **Hold**
- The stock is currently in an uptrend, but the high debt-to-equity ratio and potential overvaluation may make it a higher-risk investment. It's recommended to hold the stock and monitor its performance before making any further decisions.
